In [ ]:

# Step 68: Keep only videos that OpenCV can read

valid_videos_df = download_report[
    (download_report["downloaded"] == True) &
    (download_report["readable"] == True) &
    (download_report["frame_count"] > 0)
].copy()

# Keep only one record per gloss and video
valid_videos_df = valid_videos_df.drop_duplicates(
    subset=["gloss", "video_id"]
).reset_index(drop=True)

print("VERIFIED VIDEO MANIFEST")
print(
    valid_videos_df[
        ["gloss", "video_id", "signer_id", "frame_count", "filepath"]
    ].to_string(index=False)
)

print("\nVIDEOS PER CLASS")
print(
    valid_videos_df.groupby("gloss")["video_id"]
    .nunique()
    .to_string()
)

print("\nTOTAL VERIFIED VIDEOS:", len(valid_videos_df))
print("CLASSES REPRESENTED:", valid_videos_df["gloss"].nunique())


In [ ]:

# Step 69: Prioritize sources that returned usable videos

preferred_domains = [
    "aslbricks.org",
    "s3-us-west-1.amazonaws.com",
    "signstock.blob.core.windows.net"
]

# Start from the five selected classes
more_candidates = experiment_df[
    experiment_df["domain"].isin(preferred_domains)
].copy()

# Exclude videos already in our verified manifest
verified_ids = set(
    valid_videos_df["video_id"].astype(str)
)

more_candidates["video_id"] = (
    more_candidates["video_id"].astype(str)
)

more_candidates = more_candidates[
    ~more_candidates["video_id"].isin(verified_ids)
].copy()

# Keep direct MP4 URLs only
more_candidates["clean_url"] = (
    more_candidates["url"].apply(get_clean_url)
)

more_candidates["is_direct_mp4"] = (
    more_candidates["clean_url"].apply(
        lambda url: urlparse(url).path.lower().endswith(".mp4")
    )
)

more_candidates = more_candidates[
    more_candidates["is_direct_mp4"]
].drop_duplicates(subset=["gloss", "video_id"])

# Show up to 10 new candidates per class
candidate_sample = (
    more_candidates.groupby("gloss", group_keys=False)
    .head(10)
)

print("ADDITIONAL CANDIDATES BY CLASS")
print(
    candidate_sample.groupby("gloss")
    .agg(
        candidates=("video_id", "nunique"),
        sources=("domain", "nunique")
    )
    .to_string()
)

print("\nCANDIDATE URLS")
print(
    candidate_sample[
        ["gloss", "video_id", "signer_id", "domain", "clean_url"]
    ].to_string(index=False)
)


In [ ]:

# Step 70: Clean URLs, download, and validate new candidates

import os
import re
import requests
import cv2
import pandas as pd
from urllib.parse import urlparse, unquote

def normalize_url(value):
    value = str(value).strip()

    # Extract URL from Markdown links
    match = re.search(r"\]\((https?://.*)\)", value)
    if match:
        value = match.group(1)

    # Remove Markdown escape characters
    value = value.replace(r"\(", "(").replace(r"\)", ")")

    return value

candidate_sample = candidate_sample.copy()
candidate_sample["download_url"] = (
    candidate_sample["url"].apply(normalize_url)
)

DOWNLOAD_PATH = "/content/asl_dataset/expanded_sample"
os.makedirs(DOWNLOAD_PATH, exist_ok=True)

new_records = []

for _, row in candidate_sample.iterrows():
    gloss = row["gloss"]
    video_id = str(row["video_id"])
    url = row["download_url"]

    filepath = os.path.join(
        DOWNLOAD_PATH, f"{gloss}_{video_id}.mp4"
    )

    record = {
        "gloss": gloss,
        "video_id": video_id,
        "signer_id": str(row["signer_id"]),
        "filepath": filepath,
        "downloaded": False,
        "readable": False,
        "frame_count": 0,
        "error": ""
    }

    try:
        response = requests.get(
            url,
            timeout=30,
            allow_redirects=True
        )
        response.raise_for_status()

        content_type = response.headers.get(
            "content-type", ""
        ).lower()

        content = response.content

        # Reject obvious HTML responses
        if (
            "text/html" in content_type
            or content.lstrip().lower().startswith(
                (b"<!doctype html", b"<html")
            )
        ):
            record["error"] = "Server returned HTML, not video"
        elif len(content) < 1024:
            record["error"] = "Response too small to be a usable video"
        else:
            with open(filepath, "wb") as file:
                file.write(content)

            record["downloaded"] = True

            cap = cv2.VideoCapture(filepath)
            record["readable"] = cap.isOpened()
            record["frame_count"] = int(
                cap.get(cv2.CAP_PROP_FRAME_COUNT)
            )
            cap.release()

            if record["frame_count"] <= 0:
                record["readable"] = False
                record["error"] = "OpenCV found no video frames"

            if not record["readable"]:
                os.remove(filepath)

    except Exception as error:
        record["error"] = str(error)[:150]

    new_records.append(record)

new_download_report = pd.DataFrame(new_records)

print("NEW VIDEO VALIDATION REPORT")
print(
    new_download_report[
        ["gloss", "video_id", "downloaded",
         "readable", "frame_count", "error"]
    ].to_string(index=False)
)

print("\nREADABLE VIDEOS BY CLASS")
print(
    new_download_report[
        new_download_report["readable"]
    ].groupby("gloss")["video_id"]
    .nunique()
    .to_string()
)

print("\nTOTAL READABLE NEW VIDEOS:",
      int(new_download_report["readable"].sum()))



In [ ]:

# Step 71: Merge all verified videos

old_valid = valid_videos_df.copy()
new_valid = new_download_report[
    (new_download_report["downloaded"] == True) &
    (new_download_report["readable"] == True) &
    (new_download_report["frame_count"] > 0)
].copy()

# Keep consistent column names
required_columns = [
    "gloss", "video_id", "signer_id",
    "filepath", "frame_count"
]

old_valid = old_valid[required_columns]
new_valid = new_valid[required_columns]

# Merge the two verified batches
final_candidate_df = pd.concat(
    [old_valid, new_valid],
    ignore_index=True
)

final_candidate_df["video_id"] = (
    final_candidate_df["video_id"].astype(str)
)
final_candidate_df["signer_id"] = (
    final_candidate_df["signer_id"].astype(str)
)

# Remove duplicate gloss-video entries
final_candidate_df = final_candidate_df.drop_duplicates(
    subset=["gloss", "video_id"]
).reset_index(drop=True)

print("TOTAL VERIFIED VIDEOS:", len(final_candidate_df))

print("\nVIDEOS PER CLASS")
print(
    final_candidate_df.groupby("gloss")
    .agg(
        videos=("video_id", "nunique"),
        signers=("signer_id", "nunique")
    )
    .to_string()
)

print("\nSIGNER OVERLAP BETWEEN CLASSES")
signer_overlap = pd.crosstab(
    final_candidate_df["signer_id"],
    final_candidate_df["gloss"]
)

print(signer_overlap.to_string())

print("\nFILE VALIDATION")
missing_files = final_candidate_df[
    ~final_candidate_df["filepath"].apply(os.path.isfile)
]

print("Missing files:", len(missing_files))


In [ ]:

# Step 72: Split by signer, not by video

from sklearn.model_selection import GroupShuffleSplit

dataset_df = final_candidate_df.copy()

# First, reserve unseen signers for the test set
splitter_test = GroupShuffleSplit(
    n_splits=100,
    test_size=0.25,
    random_state=42
)

best_split = None
best_score = -1

# Try several group splits and prefer those covering all classes
for train_idx, test_idx in splitter_test.split(
    dataset_df,
    groups=dataset_df["signer_id"]
):
    train_temp = dataset_df.iloc[train_idx]
    test_temp = dataset_df.iloc[test_idx]

    score = (
        len(set(train_temp["gloss"])) +
        len(set(test_temp["gloss"]))
    )

    if score > best_score:
        best_score = score
        best_split = (train_idx, test_idx)

train_val_idx, test_idx = best_split

train_val_df = dataset_df.iloc[train_val_idx].copy()
test_df = dataset_df.iloc[test_idx].copy()

# Split the remaining training signers into train and validation groups
splitter_val = GroupShuffleSplit(
    n_splits=100,
    test_size=0.25,
    random_state=43
)

best_val_split = None
best_val_score = -1

for train_idx, val_idx in splitter_val.split(
    train_val_df,
    groups=train_val_df["signer_id"]
):
    train_temp = train_val_df.iloc[train_idx]
    val_temp = train_val_df.iloc[val_idx]

    score = (
        len(set(train_temp["gloss"])) +
        len(set(val_temp["gloss"]))
    )

    if score > best_val_score:
        best_val_score = score
        best_val_split = (train_idx, val_idx)

train_idx, val_idx = best_val_split

train_df = train_val_df.iloc[train_idx].copy()
val_df = train_val_df.iloc[val_idx].copy()

print("TRAIN VIDEOS:", len(train_df))
print("VALIDATION VIDEOS:", len(val_df))
print("TEST VIDEOS:", len(test_df))

print("\nCLASSES IN EACH SPLIT")
print("Train:", sorted(train_df["gloss"].unique()))
print("Validation:", sorted(val_df["gloss"].unique()))
print("Test:", sorted(test_df["gloss"].unique()))

print("\nSIGNER OVERLAP CHECK")
print("Train ∩ Validation:", set(train_df["signer_id"]) & set(val_df["signer_id"]))
print("Train ∩ Test:", set(train_df["signer_id"]) & set(test_df["signer_id"]))
print("Validation ∩ Test:", set(val_df["signer_id"]) & set(test_df["signer_id"]))

print("\nCLASS COUNTS")
print("TRAIN\n", train_df["gloss"].value_counts().to_string())
print("VALIDATION\n", val_df["gloss"].value_counts().to_string())
print("TEST\n", test_df["gloss"].value_counts().to_string())


In [ ]:

# Step 73: Inspect the current split by signer and class

for split_name, split_df in [
    ("TRAIN", train_df),
    ("VALIDATION", val_df),
    ("TEST", test_df)
]:
    print(f"\n{'=' * 15} {split_name} {'=' * 15}")

    print("\nVideos per class:")
    print(
        split_df.groupby("gloss")["video_id"]
        .nunique()
        .to_string()
    )

    print("\nVideos per signer:")
    print(
        split_df.groupby("signer_id")
        .agg(
            videos=("video_id", "count"),
            classes=("gloss", "nunique")
        )
        .sort_values(["videos", "classes"], ascending=False)
        .to_string()
    )

print("\nALL DATA: SIGNERS PER CLASS")
print(
    dataset_df.groupby("gloss")["signer_id"]
    .nunique()
    .to_string()
)


In [ ]:

# Step 74: Measure how much candidate data remains

verified_keys = set(
    zip(
        final_candidate_df["gloss"].astype(str),
        final_candidate_df["video_id"].astype(str)
    )
)

remaining_df = experiment_df.copy()

remaining_df["gloss"] = remaining_df["gloss"].astype(str)
remaining_df["video_id"] = remaining_df["video_id"].astype(str)

remaining_df = remaining_df[
    ~remaining_df.apply(
        lambda row: (row["gloss"], row["video_id"]) in verified_keys,
        axis=1
    )
].copy()

remaining_df["clean_url"] = (
    remaining_df["url"].apply(get_clean_url)
)

remaining_df["direct_mp4"] = remaining_df["clean_url"].apply(
    lambda url: urlparse(url).path.lower().endswith(".mp4")
)

remaining_direct = remaining_df[
    remaining_df["direct_mp4"]
].copy()

print("REMAINING DIRECT MP4 CANDIDATES")
print(
    remaining_direct.groupby("gloss")
    .agg(
        candidate_videos=("video_id", "nunique"),
        candidate_signers=("signer_id", "nunique"),
        sources=("domain", "nunique")
    )
    .to_string()
)

print("\nCANDIDATES BY CLASS AND SOURCE")
print(
    remaining_direct.groupby(["gloss", "domain"])
    .agg(videos=("video_id", "nunique"))
    .to_string()
)


In [ ]:

# Step 75: Test remaining candidate URLs before downloading

import requests
import pandas as pd

test_candidates = remaining_direct.copy()

# Normalize URLs and avoid duplicate gloss/video entries
test_candidates["test_url"] = (
    test_candidates["url"].apply(normalize_url)
)

test_candidates = test_candidates.drop_duplicates(
    subset=["gloss", "video_id"]
)

def check_candidate_url(url):
    try:
        response = requests.get(
            url,
            headers={"Range": "bytes=0-1023"},
            timeout=15,
            stream=True,
            allow_redirects=True
        )

        status = response.status_code
        content_type = response.headers.get(
            "content-type", ""
        ).lower()
        final_url = response.url
        response.close()

        ok = (
            status in [200, 206]
            and "text/html" not in content_type
            and "/lander" not in urlparse(final_url).path.lower()
        )

        return pd.Series({
            "http_status": status,
            "content_type": content_type,
            "url_ok": ok
        })

    except Exception as error:
        return pd.Series({
            "http_status": "ERROR",
            "content_type": str(error)[:100],
            "url_ok": False
        })

url_results = test_candidates["test_url"].apply(
    check_candidate_url
)

tested_candidates = pd.concat(
    [test_candidates.reset_index(drop=True),
     url_results.reset_index(drop=True)],
    axis=1
)

print("URL CHECK RESULTS BY CLASS")
print(
    tested_candidates.groupby("gloss")["url_ok"]
    .agg(["sum", "count"])
    .to_string()
)

print("\nCANDIDATES THAT PASSED THE URL CHECK")
print(
    tested_candidates[tested_candidates["url_ok"]][
        ["gloss", "video_id", "signer_id",
         "domain", "http_status"]
    ].to_string(index=False)
)


In [ ]:

# Step 76: Download and validate passing candidates

import os
import requests
import cv2
import pandas as pd

DOWNLOAD_PATH = "/content/asl_dataset/expanded_sample"
os.makedirs(DOWNLOAD_PATH, exist_ok=True)

passing_candidates = tested_candidates[
    tested_candidates["url_ok"] == True
].copy()

extra_records = []

for _, row in passing_candidates.iterrows():
    gloss = str(row["gloss"])
    video_id = str(row["video_id"])
    signer_id = str(row["signer_id"])
    url = row["test_url"]

    filepath = os.path.join(
        DOWNLOAD_PATH, f"{gloss}_{video_id}.mp4"
    )

    record = {
        "gloss": gloss,
        "video_id": video_id,
        "signer_id": signer_id,
        "filepath": filepath,
        "downloaded": False,
        "readable": False,
        "frame_count": 0,
        "error": ""
    }

    try:
        response = requests.get(
            url,
            timeout=30,
            allow_redirects=True
        )
        response.raise_for_status()

        content = response.content
        content_type = response.headers.get(
            "content-type", ""
        ).lower()

        if (
            len(content) < 1024
            or "text/html" in content_type
            or content.lstrip().lower().startswith(
                (b"<!doctype html", b"<html")
            )
        ):
            record["error"] = "Not a usable video response"
        else:
            with open(filepath, "wb") as file:
                file.write(content)

            record["downloaded"] = True

            cap = cv2.VideoCapture(filepath)
            record["readable"] = cap.isOpened()
            record["frame_count"] = int(
                cap.get(cv2.CAP_PROP_FRAME_COUNT)
            )
            cap.release()

            if record["frame_count"] <= 0:
                record["readable"] = False
                record["error"] = "No readable video frames"

            if not record["readable"]:
                os.remove(filepath)

    except Exception as error:
        record["error"] = str(error)[:150]

    extra_records.append(record)

extra_download_report = pd.DataFrame(extra_records)

print("ADDITIONAL VIDEO VALIDATION")
print(
    extra_download_report[
        ["gloss", "video_id", "downloaded",
         "readable", "frame_count", "error"]
    ].to_string(index=False)
)

print("\nREADABLE NEW VIDEOS BY CLASS")
print(
    extra_download_report[
        extra_download_report["readable"]
    ].groupby("gloss")["video_id"]
    .nunique()
    .to_string()
)

print("\nTOTAL NEW READABLE VIDEOS:",
      int(extra_download_report["readable"].sum()))


In [ ]:

# Step 77: Merge all three verified batches

newest_valid = extra_download_report[
    (extra_download_report["downloaded"] == True) &
    (extra_download_report["readable"] == True) &
    (extra_download_report["frame_count"] > 0)
].copy()

columns = [
    "gloss", "video_id", "signer_id",
    "filepath", "frame_count"
]

combined_df = pd.concat(
    [
        final_candidate_df[columns],
        newest_valid[columns]
    ],
    ignore_index=True
)

combined_df["gloss"] = combined_df["gloss"].astype(str)
combined_df["video_id"] = combined_df["video_id"].astype(str)
combined_df["signer_id"] = combined_df["signer_id"].astype(str)

# Remove duplicate annotations for the same gloss and video
combined_df = combined_df.drop_duplicates(
    subset=["gloss", "video_id"]
).reset_index(drop=True)

# Confirm the files still exist
combined_df["file_exists"] = combined_df["filepath"].apply(
    os.path.isfile
)

combined_df = combined_df[
    combined_df["file_exists"]
].drop(columns=["file_exists"]).reset_index(drop=True)

print("TOTAL VERIFIED VIDEOS:", len(combined_df))

print("\nCLASS AND SIGNER SUMMARY")
print(
    combined_df.groupby("gloss")
    .agg(
        videos=("video_id", "nunique"),
        signers=("signer_id", "nunique")
    )
    .to_string()
)

print("\nTOTAL DISTINCT SIGNERS:",
      combined_df["signer_id"].nunique())

print("\nSIGNERS PER CLASS")
print(
    combined_df.groupby("gloss")["signer_id"]
    .nunique()
    .to_string()
)

print("\nVIDEO COUNT BALANCE")
print(
    combined_df["gloss"].value_counts().sort_index().to_string()
)


In [ ]:

import os
import hashlib
import pandas as pd

def get_file_hash(filepath):
    sha256 = hashlib.sha256()

    with open(filepath, "rb") as file:
        for chunk in iter(lambda: file.read(8192), b""):
            sha256.update(chunk)

    return sha256.hexdigest()

combined_df["file_hash"] = combined_df["filepath"].apply(
    get_file_hash
)

print("TOTAL VIDEO FILES:", len(combined_df))
print("UNIQUE FILE CONTENTS:", combined_df["file_hash"].nunique())

duplicate_files = combined_df[
    combined_df.duplicated("file_hash", keep=False)
].sort_values("file_hash")

if len(duplicate_files) > 0:
    print("\nPOSSIBLE EXACT DUPLICATES:")
    print(
        duplicate_files[
            ["gloss", "video_id", "signer_id", "filepath", "file_hash"]
        ].to_string(index=False)
    )
else:
    print("\nNo exact duplicate files found.")

print("\nSAME VIDEO ID USED FOR MULTIPLE LABELS:")
print(
    combined_df.groupby("video_id")["gloss"]
    .nunique()
    .loc[lambda values: values > 1]
    .to_string()
)


In [ ]:

import cv2
import matplotlib.pyplot as plt
import math
import os

# Select up to 2 videos from each class
inspection_df = (
    combined_df.groupby("gloss", group_keys=False)
    .head(2)
    .reset_index(drop=True)
)

fig, axes = plt.subplots(
    len(inspection_df), 1,
    figsize=(8, len(inspection_df) * 3)
)

if len(inspection_df) == 1:
    axes = [axes]

for ax, (_, row) in zip(axes, inspection_df.iterrows()):
    cap = cv2.VideoCapture(row["filepath"])

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.set(cv2.CAP_PROP_POS_FRAMES, max(0, total_frames // 2))

    success, frame = cap.read()
    cap.release()

    if success:
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        ax.imshow(frame)
        ax.set_title(
            f"Label: {row['gloss']} | "
            f"Video: {row['video_id']} | "
            f"Signer: {row['signer_id']}"
        )
    else:
        ax.set_title(f"Could not read: {row['filepath']}")

    ax.axis("off")

plt.tight_layout()
plt.show()


In [ ]:

from IPython.display import display, HTML
import base64
import os

sample_videos = (
    combined_df.groupby("gloss", group_keys=False)
    .head(1)
    .reset_index(drop=True)
)

for _, row in sample_videos.iterrows():
    with open(row["filepath"], "rb") as video_file:
        video_data = base64.b64encode(video_file.read()).decode("utf-8")

    print(
        f"Label: {row['gloss']} | "
        f"Video ID: {row['video_id']} | "
        f"Signer ID: {row['signer_id']}"
    )

    display(HTML(f"""
        <video width="400" controls>
            <source src="data:video/mp4;base64,{video_data}"
                    type="video/mp4">
            Your browser does not support this video.
        </video>
    """))


In [ ]:

import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

# Keep only the required columns
split_df = combined_df[
    ["gloss", "video_id", "signer_id", "filepath", "frame_count"]
].copy()

split_df["signer_id"] = split_df["signer_id"].astype(str)

# First: reserve unseen signers for the test set
splitter_test = GroupShuffleSplit(
    n_splits=100,
    test_size=0.20,
    random_state=42
)

trainval_idx, test_idx = next(
    splitter_test.split(
        split_df,
        groups=split_df["signer_id"]
    )
)

trainval_df = split_df.iloc[trainval_idx].copy()
test_df = split_df.iloc[test_idx].copy()

# Second: reserve unseen signers for validation
splitter_val = GroupShuffleSplit(
    n_splits=100,
    test_size=0.20,
    random_state=21
)

train_idx, val_idx = next(
    splitter_val.split(
        trainval_df,
        groups=trainval_df["signer_id"]
    )
)

train_df = trainval_df.iloc[train_idx].copy()
val_df = trainval_df.iloc[val_idx].copy()

# Report split sizes and class distributions
for name, data in [
    ("TRAIN", train_df),
    ("VALIDATION", val_df),
    ("TEST", test_df)
]:
    print(f"\n{name}: {len(data)} videos")
    print("Signers:", sorted(data["signer_id"].unique()))
    print(data["gloss"].value_counts().sort_index().to_dict())

# Confirm there is no signer overlap
train_signers = set(train_df["signer_id"])
val_signers = set(val_df["signer_id"])
test_signers = set(test_df["signer_id"])

print("\nSIGNER OVERLAP CHECK")
print("Train vs validation:", train_signers & val_signers)
print("Train vs test:", train_signers & test_signers)
print("Validation vs test:", val_signers & test_signers)


In [ ]:

import cv2
import numpy as np

def preprocess_video(video_path, num_frames=8, image_size=224):
    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        raise ValueError(f"Cannot open video: {video_path}")

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if total_frames <= 0:
        cap.release()
        raise ValueError(f"No frames found: {video_path}")

    frame_indices = np.linspace(
        0, total_frames - 1, num_frames, dtype=int
    )

    frames = []

    for frame_index in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_index))
        success, frame = cap.read()

        if not success:
            cap.release()
            raise ValueError(
                f"Could not read frame {frame_index}: {video_path}"
            )

        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame = cv2.resize(frame, (image_size, image_size))
        frame = frame.astype(np.float32) / 255.0

        frames.append(frame)

    cap.release()

    return np.array(frames, dtype=np.float32)


# Test preprocessing on one training video
sample_path = train_df.iloc[0]["filepath"]
sample_frames = preprocess_video(sample_path)

print("Video path:", sample_path)
print("Processed shape:", sample_frames.shape)
print("Data type:", sample_frames.dtype)
print("Minimum pixel value:", sample_frames.min())
print("Maximum pixel value:", sample_frames.max())


In [ ]:
from sklearn.preprocessing import LabelEncoder
import numpy as np

label_encoder = LabelEncoder()
label_encoder.fit(sorted(combined_df["gloss"].unique()))

class_names = list(label_encoder.classes_)

print("Class names:", class_names)
print("Label mapping:")

for i, name in enumerate(class_names):
    print(name, "=", i)


def prepare_split(dataframe):
    X = []
    y = []

    for _, row in dataframe.iterrows():
        frames = preprocess_video(row["filepath"])
        X.append(frames)

        label = label_encoder.transform([row["gloss"]])[0]
        y.append(label)

    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int64)


X_train, y_train = prepare_split(train_df)
X_val, y_val = prepare_split(val_df)
X_test, y_test = prepare_split(test_df)

print("\nDATA PREPARATION COMPLETE")
print("Training data:", X_train.shape, y_train.shape)
print("Validation data:", X_val.shape, y_val.shape)
print("Testing data:", X_test.shape, y_test.shape)

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

tf.random.set_seed(42)
np.random.seed(42)

model = models.Sequential([
    layers.Input(shape=(8, 224, 224, 3)),

    layers.Conv3D(16, (3, 3, 3), activation="relu"),
    layers.MaxPooling3D((1, 2, 2)),

    layers.Conv3D(32, (3, 3, 3), activation="relu"),
    layers.MaxPooling3D((1, 2, 2)),

    layers.GlobalAveragePooling3D(),

    layers.Dense(64, activation="relu"),
    layers.Dropout(0.5),

    layers.Dense(5, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=4,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=15,
    batch_size=2,
    callbacks=[early_stopping],
    verbose=1
)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

test_probabilities = model.predict(X_test)
y_pred = np.argmax(test_probabilities, axis=1)

print("Test Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        labels=list(range(len(class_names))),
        target_names=class_names,
        zero_division=0
    )
)

In [ ]:
for i in range(len(y_test)):
    actual = class_names[y_test[i]]
    predicted = class_names[y_pred[i]]
    confidence = np.max(test_probabilities[i]) * 100

    print("Video:", test_df.iloc[i]["video_id"])
    print("Actual sign:", actual)
    print("Predicted sign:", predicted)
    print("Confidence:", round(confidence, 2), "%")
    print("-" * 35)

In [ ]:
print("TRAINING CLASS COUNTS")
print(train_df["gloss"].value_counts().sort_index())

print("\nVALIDATION CLASS COUNTS")
print(val_df["gloss"].value_counts().sort_index())

print("\nTEST CLASS COUNTS")
print(test_df["gloss"].value_counts().sort_index())

print("\nTRAINING LABELS")
print(y_train)

print("\nPREDICTED TEST LABELS")
print(y_pred)

In [ ]:
import os

print("DATASET FILES")
if os.path.exists("/content/asl_dataset"):
    print(os.listdir("/content/asl_dataset"))
else:
    print("Folder not found")

print("\nOUTPUT FILES")
if os.path.exists("/content/asl_outputs"):
    print(os.listdir("/content/asl_outputs"))
else:
    print("Folder not found")

print("\nMODEL FILES")
if os.path.exists("/content/asl_models"):
    print(os.listdir("/content/asl_models"))
else:
    print("Folder /content/asl_models not found (it has not been created yet)")

print("\nIMPORTANT VARIABLES")
for name in [
    "df", "full_manifest", "valid_df", "combined_df",
    "landmark_df", "train_df", "val_df", "test_df",
    "X_train_video", "X_train_landmarks", "y_train", "y_pred",
    "label_encoder", "model"
]:
    print(f"{name}: {'Available' if name in globals() else 'Not available'}")

In [ ]:
import os

folders = [
    "/content/asl_dataset",
    "/content/asl_dataset/videos",
    "/content/asl_dataset/expanded_sample",
    "/content/asl_outputs",
    "/content/asl_models"
]

for folder in folders:
    print(f"\n{folder}")
    if os.path.exists(folder):
        for item in os.listdir(folder)[:30]:
            path = os.path.join(folder, item)
            if os.path.isfile(path):
                print(f"  {item} ({os.path.getsize(path):,} bytes)")
            else:
                print(f"  {item}/")
    else:
        print("  Folder not found")

print("\nIMPORTANT VARIABLES")
for name in [
    "df", "full_manifest", "valid_df", "combined_df",
    "landmark_df", "train_df", "val_df", "test_df",
    "X_train_video", "X_train_landmarks", "y_train", "y_pred",
    "label_encoder", "model"
]:
    print(f"{name}: {'Available' if name in globals() else 'Not available'}")

In [ ]:
import os
import pandas as pd
import cv2

folders = [
    "/content/asl_dataset/videos",
    "/content/asl_dataset/expanded_sample"
]

records = []

for folder in folders:
    for filename in os.listdir(folder):
        if not filename.lower().endswith(".mp4"):
            continue

        path = os.path.join(folder, filename)

        if os.path.getsize(path) < 1000:
            continue

        cap = cv2.VideoCapture(path)
        readable = cap.isOpened()
        frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) if readable else 0
        cap.release()

        if readable and frames > 0:
            records.append({
                "video_id": os.path.splitext(filename)[0],
                "filename": filename,
                "path": path
            })

video_inventory = pd.DataFrame(records)
video_inventory = video_inventory.drop_duplicates("path")

print("Total readable video files:", len(video_inventory))
print("\nVideo files by folder:")
display(
    video_inventory["path"]
    .apply(os.path.dirname)
    .value_counts()
    .to_frame("count")
)

display(video_inventory.head(10))

In [ ]:
import json
import os
import pandas as pd

ANNOTATION_PATH = "/content/asl_dataset/WLASL_v0.3.json"

with open(ANNOTATION_PATH, "r") as f:
    annotations = json.load(f)

records = []

for item in annotations:
    for instance in item.get("instances", []):
        records.append({
            "gloss": item["gloss"],
            "video_id": str(instance["video_id"]),
            "signer_id": instance.get("signer_id"),
            "split": instance.get("split")
        })

full_manifest = pd.DataFrame(records)

# Match using video ID, regardless of the filename prefix
video_inventory["match_id"] = video_inventory["filename"].apply(
    lambda name: os.path.splitext(name)[0].split("_")[-1]
)

full_manifest["match_id"] = full_manifest["video_id"].astype(str)

matched_df = video_inventory.merge(
    full_manifest,
    on="match_id",
    how="inner"
)

matched_df = matched_df.drop_duplicates(subset="path").reset_index(drop=True)

print("Matched videos:", len(matched_df))
print("\nVideos per sign:")
display(matched_df["gloss"].value_counts().to_frame("videos"))

print("\nUnique signers:", matched_df["signer_id"].nunique())

matched_df.to_csv(
    "/content/asl_outputs/matched_video_inventory.csv",
    index=False
)

print("\nSaved matched video inventory.")